# Tabmed ML: Мобильный эмбеддер и сквозной пайплайн детекции и кластеризации

Данный блокнот содержит:
1. Валидацию дообученного метрического эмбеддера `MobileNetV3-Small` (~6 МБ) на выборке `e-PillID` (ARI = 1.000).
2. Сквозной пайплайн `TabmedPipeline`: Детекция (`best.pt`) -> Кроппер с dynamic padding -> Векторизация -> Кластеризация без учителя (`cosine`, threshold=0.38).
3. Визуализацию результатов и формирование структурированного JSON-отчета.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from ultralytics import YOLO

# 1. Настройка устройства
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Вычислительное устройство: {device}")
if device.type == "cuda":
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")

# 2. Базовые пути проекта
BASE_DIR = Path.cwd()
DETECTOR_PATH = BASE_DIR / "best.pt"
EMBEDDER_PATH = BASE_DIR / "mobile_pill_embedder.pt"

print(f"Детектор best.pt найден: {DETECTOR_PATH.is_file()}")
print(f"Эмбеддер mobile_pill_embedder.pt найден: {EMBEDDER_PATH.is_file()}")

## 1. Валидация мобильного эмбеддера на 8 препаратах e-PillID

In [ ]:
# Архитектура мобильного эмбеддера
class MobilePillEmbedder(nn.Module):
    def __init__(self, emb_dim: int = 128):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=None)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, emb_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        embeddings = self.head(features)
        return F.normalize(embeddings, p=2, dim=1)

# Загрузка весов
eval_model = MobilePillEmbedder(emb_dim=128).to(device)
eval_model.load_state_dict(torch.load(EMBEDDER_PATH, map_location=device))
eval_model.eval()

embedder_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Подготовка выборки e-PillID
DATASET_ROOT = Path(r"D:\Tabmed\ePillID_data")
CSV_PATH = DATASET_ROOT / "folds" / "pilltypeid_nih_sidelbls0.01_metric_5folds" / "base" / "pilltypeid_nih_sidelbls0.01_metric_5folds_all.csv"
IMAGES_BASE = DATASET_ROOT / "classification_data"

df = pd.read_csv(CSV_PATH)
df["full_path"] = df["image_path"].apply(lambda p: IMAGES_BASE / Path(p))
top_classes = df["pilltype_id"].value_counts().head(8).index.tolist()
sample_df = df[df["pilltype_id"].isin(top_classes)].copy()

image_paths = sample_df["full_path"].tolist()
ground_truth_labels = sample_df["pilltype_id"].tolist()

# Извлечение признаков
all_embs = []
with torch.no_grad():
    for i in range(0, len(image_paths), 16):
        batch = [embedder_transform(Image.open(p).convert("RGB")) for p in image_paths[i:i+16]]
        embs = eval_model(torch.stack(batch).to(device)).cpu().numpy()
        all_embs.append(embs)
mobile_features = np.vstack(all_embs)

# Валидация метрик
agg = AgglomerativeClustering(n_clusters=8, metric="cosine", linkage="average")
pred_agg = agg.fit_predict(mobile_features)
ari_agg = adjusted_rand_score(ground_truth_labels, pred_agg)

db = DBSCAN(eps=0.112, min_samples=2, metric="cosine")
pred_db = db.fit_predict(mobile_features)
ari_db = adjusted_rand_score(ground_truth_labels, pred_db)

print("=== Результаты валидации мобильного эмбеддера ===")
print(f"AgglomerativeClustering (k=8): ARI = {ari_agg:.3f}")
print(f"DBSCAN (eps=0.112):            ARI = {ari_db:.3f}")

In [ ]:
# Визуализация сформированных групп на эталонном датасете
unique_clusters = sorted(list(set(pred_db)))
TARGET_H, HEADER_H, TITLE_H = 180, 45, 35
cluster_rows = []

for cl_id in unique_clusters:
    indices = [i for i, lbl in enumerate(pred_db) if lbl == cl_id]
    title = f"Кластер {cl_id} | Таблеток: {len(indices)}" if cl_id != -1 else f"Шум / Одиночные: {len(indices)}"
    tiles = []
    for img_idx in indices:
        img_p = image_paths[img_idx]
        real_drug = ground_truth_labels[img_idx].split("_")[0]
        bgr = cv2.imread(str(img_p))
        if bgr is None: continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        new_w = int(w * (TARGET_H / h))
        resized = cv2.resize(rgb, (new_w, TARGET_H), interpolation=cv2.INTER_AREA)
        tile = np.full((TARGET_H + HEADER_H, new_w, 3), 255, dtype=np.uint8)
        tile[HEADER_H:, :] = resized
        cv2.putText(tile, f"NDC: {real_drug}", (4, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(tile, f"{img_p.name[:12]}...", (4, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (70, 70, 70), 1, cv2.LINE_AA)
        tiles.append(tile)
    if not tiles: continue
    row_img = np.hstack(tiles)
    title_bar = np.full((TITLE_H, row_img.shape[1], 3), 245, dtype=np.uint8)
    cv2.putText(title_bar, title, (15, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2, cv2.LINE_AA)
    cluster_rows.append(np.vstack([title_bar, row_img]))

if cluster_rows:
    max_w = max(r.shape[1] for r in cluster_rows)
    canvas = np.vstack([np.hstack([r, np.full((r.shape[0], max_w - r.shape[1], 3), 255, dtype=np.uint8)]) if r.shape[1] < max_w else r for r in cluster_rows])
    fig_w = 16
    fig_h = max(4, int(fig_w * (canvas.shape[0] / canvas.shape[1])))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(canvas)
    ax.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.show()

## 2. Сквозной пайплайн (End-to-End Pipeline)

In [ ]:
class TabmedPipeline:
    def __init__(
        self,
        detector_path: Path = DETECTOR_PATH,
        embedder_path: Path = EMBEDDER_PATH,
        device: torch.device = device,
    ):
        if not detector_path.is_file():
            raise FileNotFoundError(f"Детектор не найден: {detector_path}")
        if not embedder_path.is_file():
            raise FileNotFoundError(f"Эмбеддер не найден: {embedder_path}")

        self.device = device
        self.detector = YOLO(str(detector_path))
        self.embedder = MobilePillEmbedder(emb_dim=128).to(device)
        self.embedder.load_state_dict(torch.load(embedder_path, map_location=device))
        self.embedder.eval()

    def crop_detections(
        self, img_bgr: np.ndarray, boxes: np.ndarray, classes: np.ndarray, confidences: np.ndarray, padding_ratio: float = 0.08
    ) -> list[dict]:
        h_img, w_img = img_bgr.shape[:2]
        crops_data = []
        for idx, (b, c, cf) in enumerate(zip(boxes, classes, confidences)):
            x1, y1, x2, y2 = b
            pw, ph = int((x2 - x1) * padding_ratio), int((y2 - y1) * padding_ratio)
            cx1, cy1 = max(0, int(x1 - pw)), max(0, int(y1 - ph))
            cx2, cy2 = min(w_img, int(x2 + pw)), min(h_img, int(y2 + ph))
            crop_bgr = img_bgr[cy1:cy2, cx1:cx2]
            if crop_bgr.size == 0: continue
            crop_pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
            crops_data.append({
                "crop_id": idx,
                "box": [round(float(v), 2) for v in b],
                "class_name": "capsule" if int(c) == 0 else "tablet",
                "conf": round(float(cf), 4),
                "crop_pil": crop_pil,
                "crop_bgr": crop_bgr,
            })
        return crops_data

    def extract_features(self, crops_data: list[dict]) -> np.ndarray:
        if not crops_data:
            return np.empty((0, 128))
        tensors = [embedder_transform(c["crop_pil"]) for c in crops_data]
        batch = torch.stack(tensors).to(self.device)
        with torch.no_grad():
            embs = self.embedder(batch).cpu().numpy()
        return embs

    def process(
        self, image_path: Path, conf_thresh: float = 0.35, dist_thresh: float = 0.38, padding_ratio: float = 0.08
    ) -> tuple[dict, np.ndarray, list[dict]]:
        img_bgr = cv2.imread(str(image_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Не удалось прочитать изображение: {image_path}")

        # 1. Детекция
        results = self.detector.predict(source=img_bgr, conf=conf_thresh, verbose=False)[0]
        boxes = results.boxes.xyxy.cpu().numpy()
        classes = results.boxes.cls.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()

        if len(boxes) == 0:
            return ({
                "source_image": image_path.name,
                "total_detected": 0,
                "total_unique_drugs": 0,
                "drug_groups": []
            }, img_bgr, [])

        # 2. Нарезка кропов
        crops_data = self.crop_detections(img_bgr, boxes, classes, confs, padding_ratio=padding_ratio)

        # 3. Векторизация
        embs = self.extract_features(crops_data)

        # 4. Кластеризация без учителя
        if len(crops_data) == 1:
            labels = [0]
        else:
            cl = AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=dist_thresh,
                metric="cosine",
                linkage="average"
            )
            labels = cl.fit_predict(embs).tolist()

        for it, lbl in zip(crops_data, labels):
            it["cluster_id"] = int(lbl)

        # 5. Сборка JSON отчета
        unique_groups = sorted(list(set(labels)))
        groups_report = []
        for gid in unique_groups:
            group_items = [it for it in crops_data if it["cluster_id"] == gid]
            classes_in_group = [it["class_name"] for it in group_items]
            dominant_cls = max(set(classes_in_group), key=classes_in_group.count)
            groups_report.append({
                "group_id": gid,
                "dominant_form": dominant_cls,
                "count": len(group_items),
                "items": [
                    {"crop_id": it["crop_id"], "bbox": it["box"], "confidence": it["conf"]}
                    for it in group_items
                ]
            })

        report = {
            "source_image": image_path.name,
            "total_detected": len(crops_data),
            "total_unique_drugs": len(unique_groups),
            "drug_groups": groups_report
        }
        return report, img_bgr, crops_data

In [ ]:
def visualize_pipeline_result(img_bgr: np.ndarray, crops_data: list[dict], report: dict):
    if not crops_data or not report.get("drug_groups"):
        print("Нет обнаруженных объектов для отображения.")
        return

    target_h, header_h, title_h = 160, 40, 32
    cluster_rows = []

    for grp in report["drug_groups"]:
        gid = grp["group_id"]
        group_items = [c for c in crops_data if c["cluster_id"] == gid]
        group_title = f"Группа {gid} | Форма: {grp['dominant_form']} | Найдено: {grp['count']} шт."

        tiles = []
        for item in group_items:
            crop_rgb = cv2.cvtColor(item["crop_bgr"], cv2.COLOR_BGR2RGB)
            h, w = crop_rgb.shape[:2]
            new_w = max(1, int(w * (target_h / h)))
            resized = cv2.resize(crop_rgb, (new_w, target_h), interpolation=cv2.INTER_AREA)

            tile = np.full((target_h + header_h, new_w, 3), 255, dtype=np.uint8)
            tile[header_h:, :] = resized
            cv2.putText(tile, f"ID: {item['crop_id']} | {item['class_name']}", (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(tile, f"Conf: {item['conf']:.2f}", (4, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (80, 80, 80), 1, cv2.LINE_AA)
            tiles.append(tile)

        if not tiles: continue
        row_img = np.hstack(tiles)
        title_bar = np.full((title_h, row_img.shape[1], 3), 240, dtype=np.uint8)
        cv2.putText(title_bar, group_title, (12, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)
        cluster_rows.append(np.vstack([title_bar, row_img]))

    if cluster_rows:
        max_w = max(r.shape[1] for r in cluster_rows)
        canvas = np.vstack([np.hstack([r, np.full((r.shape[0], max_w - r.shape[1], 3), 255, dtype=np.uint8)]) if r.shape[1] < max_w else r for r in cluster_rows])
        fig_w = 16
        fig_h = max(3, int(fig_w * (canvas.shape[0] / canvas.shape[1])))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.imshow(canvas)
        ax.axis("off")
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        plt.show()

## 3. Сквозной тест на реальном изображении

In [ ]:
import json

pipe = TabmedPipeline()
TEST_IMAGE = BASE_DIR / "test.jpg"

report, img_bgr, crops = pipe.process(
    TEST_IMAGE,
    conf_thresh=0.35,
    dist_thresh=0.38
)

print("=== ИТОГОВЫЙ JSON-ОТЧЕТ TABMED ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

visualize_pipeline_result(img_bgr, crops, report)